In [1]:
from google.colab import files
upload = files.upload()

Saving juice_data.csv to juice_data.csv


In [2]:
import numpy as np
import pandas as pd
df= pd.read_csv("juice_data.csv", header = None )
df.head()

,0,1,2,3,4,5,6,7,8,9,...,98,99,100,101,102,103,104,105,106,107
0,NaN,NaN,NaN,NaN,"Phase in degrees,",pt. no.,0.0,1.000000,2.000000,3.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,decade points,0.0,0.080000,0.160000,0.240000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,Frequency,100.0,120.226444,144.543977,173.780083,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,csv no.,Sr. No.,Sample_no.,Fruit,NaN,Time(min),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E498x002.csv,1,1,Mosambi,Impedance,0,73.8,65.900000,59.000000,53.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Drop columns from index 51 to 107 (inclusive)
df = df.drop(columns=df.columns[57:108])
df = df.iloc[4:].reset_index(drop=True)

In [4]:
columns = [
    "csv_no", "Sr_No", "Sample_no", "Fruit", "Type", "Time"
]

# Add frequency columns dynamically based on remaining columns
num_extra_cols = df.shape[1] - len(columns)

freq_cols = [f"F_{i}" for i in range(num_extra_cols)]

# Final column names
df.columns = columns + freq_cols

# Check result
print(df.head())

         csv_no Sr_No Sample_no    Fruit       Type Time   F_0   F_1   F_2  \
0  E498x002.csv     1         1  Mosambi  Impedance    0  73.8  65.9  59.0   
1  E498x002.csv     2         1  Mosambi      Phase    0 -58.8 -57.1 -55.0   
2  E498x003.csv     3         1  Mosambi  Impedance    2  77.7  70.2  63.8   
3  E498x003.csv     4         1  Mosambi      Phase    2 -53.4 -51.1 -48.5   
4  E498x004.csv     5         1  Mosambi  Impedance    4  79.2  71.5  64.8   

    F_3  ...   F_41   F_42   F_43   F_44   F_45   F_46   F_47   F_48   F_49  \
0  53.0  ...  23.60  23.60  23.60  23.60  23.60  23.60  23.60  23.60  23.70   
1 -52.5  ...  -0.12  -0.06   0.00   0.06   0.13   0.21   0.29   0.39   0.50   
2  58.2  ...  31.40  31.40  31.40  31.40  31.40  31.40  31.40  31.40  31.40   
3 -45.7  ...  -0.19  -0.15  -0.12  -0.09  -0.06  -0.03   0.00   0.04   0.07   
4  59.1  ...  31.50  31.50  31.50  31.50  31.50  31.50  31.50  31.60  31.60   

    F_50  
0  23.70  
1   0.63  
2  31.50  
3   0.11  
4

In [5]:
df = df.dropna(axis=1, how='all')

In [6]:
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

In [7]:
df.head()
print(df.shape)

(736, 57)


In [8]:
df_t4 = df[df["Time"] == 4].reset_index(drop=True)
print("t=4 shape:", df_t4.shape)


t=4 shape: (180, 57)


In [9]:
df_t4.head()

,csv_no,Sr_No,Sample_no,Fruit,Type,Time,F_0,F_1,F_2,F_3,...,F_41,F_42,F_43,F_44,F_45,F_46,F_47,F_48,F_49,F_50
0,E498x004.csv,5,1,Mosambi,Impedance,4.0,79.20,71.50,64.80,59.10,...,31.50,31.50,31.50,31.50,31.50,31.50,31.50,31.60,31.60,31.60
1,E498x004.csv,6,1,Mosambi,Phase,4.0,-53.80,-51.60,-49.00,-46.30,...,-0.18,-0.14,-0.11,-0.08,-0.05,-0.02,0.01,0.05,0.09,0.13
2,E498x009.csv,13,2,Mosambi,Impedance,4.0,84.72,77.24,70.93,65.52,...,40.44,40.44,40.44,40.44,40.44,40.45,40.46,40.49,40.52,40.57
3,E498x009.csv,14,2,Mosambi,Phase,4.0,-48.94,-46.32,-43.51,-40.51,...,-0.15,-0.13,-0.10,-0.08,-0.05,-0.03,0.00,0.03,0.06,0.10
4,E498x013.csv,21,3,Mosambi,Impedance,4.0,84.61,76.83,70.21,64.51,...,37.90,37.89,37.89,37.89,37.90,37.90,37.92,37.94,37.97,38.01


In [10]:
# Separate analysis on t=4
imp_df = df_t4[df_t4["Type"] == "Impedance"]
phase_df = df_t4[df_t4["Type"] == "Phase"]

In [11]:
drop_cols = ["csv_no", "Sr_No", "Sample_no", "Fruit", "Type", "Time"]

X_imp = imp_df.drop(columns=drop_cols)
y_imp = imp_df["Fruit"]

In [12]:

X_phase = phase_df.drop(columns=drop_cols)
y_phase = phase_df["Fruit"]

In [13]:
df_t4_merged = imp_df.merge(
    phase_df,
    on=["csv_no", "Sample_no", "Fruit", "Time"],
    suffixes=('_imp', '_phase')
)

print(df_t4_merged.head())

# Define drop_cols specific to the merged dataframe, accounting for suffixes
drop_cols_for_X = [
    "csv_no", "Sr_No_imp", "Sr_No_phase", "Sample_no", "Fruit",
    "Type_imp", "Type_phase", "Time"
]

# Ensure we only drop columns that actually exist in df_t2_merged
X = df_t4_merged.drop(columns=[col for col in drop_cols_for_X if col in df_t4_merged.columns])
y = df_t4_merged["Fruit"]

         csv_no Sr_No_imp Sample_no    Fruit   Type_imp  Time  F_0_imp  \
0  E498x004.csv         5         1  Mosambi  Impedance   4.0    79.20   
1  E498x009.csv        13         2  Mosambi  Impedance   4.0    84.72   
2  E498x013.csv        21         3  Mosambi  Impedance   4.0    84.61   
3  E498x017.csv        29         4  Mosambi  Impedance   4.0    82.69   
4  E498x021.csv        37         5  Mosambi  Impedance   4.0    84.94   

   F_1_imp  F_2_imp  F_3_imp  ...  F_41_phase  F_42_phase  F_43_phase  \
0    71.50    64.80    59.10  ...       -0.18       -0.14       -0.11   
1    77.24    70.93    65.52  ...       -0.15       -0.13       -0.10   
2    76.83    70.21    64.51  ...       -0.14       -0.11       -0.08   
3    75.23    68.92    63.52  ...       -0.20       -0.18       -0.15   
4    77.35    71.01    65.61  ...       -0.20       -0.17       -0.14   

   F_44_phase  F_45_phase  F_46_phase  F_47_phase  F_48_phase  F_49_phase  \
0       -0.08       -0.05       -0.02  

In [14]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique y values:", y.unique())
print(X.head())

X shape: (90, 102)
y shape: (90,)
Unique y values: ['Mosambi' 'Apple' 'Mango' 'Guava' 'MangoMz']
   F_0_imp  F_1_imp  F_2_imp  F_3_imp  F_4_imp  F_5_imp  F_6_imp  F_7_imp  \
0    79.20    71.50    64.80    59.10    54.20    49.90    46.40    43.50   
1    84.72    77.24    70.93    65.52    60.94    57.08    53.87    51.23   
2    84.61    76.83    70.21    64.51    59.68    55.59    52.18    49.38   
3    82.69    75.23    68.92    63.52    58.95    55.11    51.91    49.29   
4    84.94    77.35    71.01    65.61    61.06    57.25    54.10    51.53   

   F_8_imp  F_9_imp  ...  F_41_phase  F_42_phase  F_43_phase  F_44_phase  \
0    41.10    39.10  ...       -0.18       -0.14       -0.11       -0.08   
1    49.09    47.36  ...       -0.15       -0.13       -0.10       -0.08   
2    47.09    45.25  ...       -0.14       -0.11       -0.08       -0.04   
3    47.16    45.45  ...       -0.20       -0.18       -0.15       -0.12   
4    49.45    47.79  ...       -0.20       -0.17       -0.14

In [15]:
for i in range(51):
    X[f"F_{i}_real"] = X[f"F_{i}_imp"] * np.cos(X[f"F_{i}_phase"])
    X[f"F_{i}_imag"] = X[f"F_{i}_imp"] * np.sin(X[f"F_{i}_phase"])

/tmp/ipykernel_1027/3057369963.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f"F_{i}_real"] = X[f"F_{i}_imp"] * np.cos(X[f"F_{i}_phase"])
/tmp/ipykernel_1027/3057369963.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f"F_{i}_imag"] = X[f"F_{i}_imp"] * np.sin(X[f"F_{i}_phase"])
/tmp/ipykernel_1027/3057369963.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=

In [16]:
X.drop(columns=[f"F_{i}_imp" for i in range(51)] +
                [f"F_{i}_phase" for i in range(51)], inplace=True)

In [17]:
X.head()

,F_0_real,F_0_imag,F_1_real,F_1_imag,F_2_real,F_2_imag,F_3_real,F_3_imag,F_4_real,F_4_imag,...,F_46_real,F_46_imag,F_47_real,F_47_imag,F_48_real,F_48_imag,F_49_real,F_49_imag,F_50_real,F_50_imag
0,-73.164413,30.325050,16.737180,-69.513429,19.478397,61.803172,-40.150888,-43.367225,42.066042,34.177304,...,31.493700,-0.629958,31.498425,0.314995,31.560508,1.579342,31.472106,2.840162,31.333356,4.096439
1,20.575168,82.183580,-53.597691,-55.617489,63.164980,32.268409,-61.969722,-21.274962,58.051696,18.538722,...,40.431799,-1.213318,40.460000,0.000000,40.471781,1.214518,40.447086,2.429742,40.367319,4.050242
2,84.522501,3.846944,-60.406146,47.475745,28.827041,-64.019105,-10.373187,63.670535,4.168722,-59.534227,...,37.892420,0.757949,37.851764,2.273835,37.750458,3.787680,37.543638,5.674166,37.252331,7.551421
3,57.878061,59.057312,-73.181656,-17.435543,68.267599,-9.460516,-60.696183,18.728688,55.940936,-18.593390,...,38.475542,-2.697696,38.549140,-1.542789,38.588071,-0.385894,38.602622,1.158426,38.565322,2.703990
4,41.117934,74.324418,-59.955606,-48.870725,63.556103,31.670836,-60.153858,-26.195143,54.184931,28.148123,...,41.137068,-1.646361,41.170000,0.000000,41.147060,1.646761,41.004122,4.114135,40.723125,6.571876


In [18]:
num_freq = 51

freq_groups = []

for i in range(num_freq):
    real_col = X.columns.get_loc(f"F_{i}_real")
    imag_col = X.columns.get_loc(f"F_{i}_imag")

    freq_groups.append((real_col, imag_col))

In [19]:
print(freq_groups)

[(0, 1), (2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15), (16, 17), (18, 19), (20, 21), (22, 23), (24, 25), (26, 27), (28, 29), (30, 31), (32, 33), (34, 35), (36, 37), (38, 39), (40, 41), (42, 43), (44, 45), (46, 47), (48, 49), (50, 51), (52, 53), (54, 55), (56, 57), (58, 59), (60, 61), (62, 63), (64, 65), (66, 67), (68, 69), (70, 71), (72, 73), (74, 75), (76, 77), (78, 79), (80, 81), (82, 83), (84, 85), (86, 87), (88, 89), (90, 91), (92, 93), (94, 95), (96, 97), (98, 99), (100, 101)]


In [20]:
from itertools import combinations

freq_pairs = list(combinations(range(num_freq), 2))

print("Total frequency pairs:", len(freq_pairs))  # should be ~1275

Total frequency pairs: 1275


In [25]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

def apply_lda(X, y):
    X = X.fillna(X.mean())

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    lda = LinearDiscriminantAnalysis(n_components=2)
    X_lda = lda.fit_transform(X_scaled, y)

    return X_lda

In [26]:
import matplotlib.pyplot as plt

def plot_lda(X_lda, y, title):
    plt.figure(figsize=(6,5))
    for label in y.unique():
        plt.scatter(
            X_lda[y == label, 0],
            X_lda[y == label, 1],
            label=label
        )
    plt.title(title)
    plt.xlabel("LD1")
    plt.ylabel("LD2")
    plt.legend()
    plt.show()


In [27]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np

best_acc = 0
best_pair = None
best_model = None

results = []

for (f1, f2) in freq_pairs:

    # Extract column names
    cols = [
        f"F_{f1}_real", f"F_{f1}_imag",
        f"F_{f2}_real", f"F_{f2}_imag"
    ]

    X_pair = X[cols].copy()

    # Train-test split (same as reference)
    X_train, X_test, y_train, y_test = train_test_split(
        X_pair, y, test_size=0.3, random_state=42, stratify=y
    )

    # Apply LDA (same function from juice_lda_(1))
    X_train_lda = apply_lda(X_train, y_train)
    X_test_lda = apply_lda(X_test, y_test)

    # Standard Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_lda)
    X_test_scaled = scaler.transform(X_test_lda)

    # KNN with same grid
    param_grid = {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    }

    knn = KNeighborsClassifier()

    grid = GridSearchCV(
        knn,
        param_grid,
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train_scaled, y_train)

    best_knn = grid.best_estimator_

    # Predict
    y_pred = best_knn.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)

    results.append((f1, f2, acc))

    # Track best
    if acc > best_acc:
        best_acc = acc
        best_pair = (f1, f2)
        best_model = best_knn

    print(f"Pair ({f1}, {f2}) → Accuracy: {acc:.4f}")

print("\n==========================")
print(" BEST PAIR FOUND")
print("==========================")
print(f"Best Pair: {best_pair}")
print(f"Best Accuracy: {best_acc:.4f}")

Pair (0, 1) → Accuracy: 0.1481
Pair (0, 2) → Accuracy: 0.4815
Pair (0, 3) → Accuracy: 0.5185
Pair (0, 4) → Accuracy: 0.5926
Pair (0, 5) → Accuracy: 0.1852
Pair (0, 6) → Accuracy: 0.4444
Pair (0, 7) → Accuracy: 0.1481
Pair (0, 8) → Accuracy: 0.1481
Pair (0, 9) → Accuracy: 0.0370
Pair (0, 10) → Accuracy: 0.3704
Pair (0, 11) → Accuracy: 0.3704
Pair (0, 12) → Accuracy: 0.4074
Pair (0, 13) → Accuracy: 0.3704
Pair (0, 14) → Accuracy: 0.4074
Pair (0, 15) → Accuracy: 0.3333
Pair (0, 16) → Accuracy: 0.4815
Pair (0, 17) → Accuracy: 0.4444
Pair (0, 18) → Accuracy: 0.4815
Pair (0, 19) → Accuracy: 0.5185
Pair (0, 20) → Accuracy: 0.5185
Pair (0, 21) → Accuracy: 0.4815
Pair (0, 22) → Accuracy: 0.4074
Pair (0, 23) → Accuracy: 0.0741
Pair (0, 24) → Accuracy: 0.3333
Pair (0, 25) → Accuracy: 0.4444
Pair (0, 26) → Accuracy: 0.5185
Pair (0, 27) → Accuracy: 0.5185
Pair (0, 28) → Accuracy: 0.4815
Pair (0, 29) → Accuracy: 0.3333
Pair (0, 30) → Accuracy: 0.0000
Pair (0, 31) → Accuracy: 0.0370
Pair (0, 32) → Ac

In [28]:
# Extract best pair features
f1, f2 = best_pair

best_cols = [
    f"F_{f1}_real", f"F_{f1}_imag",
    f"F_{f2}_real", f"F_{f2}_imag"
]

X_best = X[best_cols].copy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_best, y, test_size=0.3, random_state=42, stratify=y
)

# LDA
X_train_lda = apply_lda(X_train, y_train)
X_test_lda = apply_lda(X_test, y_test)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_lda)
X_test_scaled = scaler.transform(X_test_lda)

# Train final KNN using best params
final_knn = best_model
final_knn.fit(X_train_scaled, y_train)

# Evaluate
y_pred = final_knn.predict(X_test_scaled)

from sklearn.metrics import classification_report, confusion_matrix

print("Final Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Final Accuracy: 0.7407407407407407

Classification Report:
               precision    recall  f1-score   support

       Apple       0.62      0.83      0.71         6
       Guava       0.60      0.50      0.55         6
       Mango       0.86      1.00      0.92         6
     MangoMz       1.00      1.00      1.00         3
     Mosambi       0.75      0.50      0.60         6

    accuracy                           0.74        27
   macro avg       0.77      0.77      0.76        27
weighted avg       0.74      0.74      0.73        27


Confusion Matrix:
 [[5 0 1 0 0]
 [2 3 0 0 1]
 [0 0 6 0 0]
 [0 0 0 3 0]
 [1 2 0 0 3]]
